In [ ]:
%%spark

# ============================================================
# 8_PUBLICAR_ORACLE
# ============================================================


def _preparar_df_oracle(df_rcm_fnc_cli_final, logger_etapa):
    tabela_oracle = TABELAS_ORACLE["oracle_cliente_recomendacao_ativa"]
    contrato = CONTRATO_ORACLE[tabela_oracle]
    tipos_oracle = {
        nome_coluna: tipo_coluna
        for nome_coluna, tipo_coluna, _ in contrato["campos"]
    }
    limites = contrato["limites"]
    view_base = registrar_view_sql(df_rcm_fnc_cli_final, "publicacao_oracle_base")

    # df = spark_sql(f"""
    #     SELECT
    #         CAST({coluna_sql(COL_NR_IDFR_PROJ)} AS {tipo_sql(tipo_campo_contrato_hive(contrato['origem_hive'], COL_NR_IDFR_PROJ))}) AS {coluna_sql(COL_NR_IDFR_PROJ)},
    #         CAST({coluna_sql(COL_NR_IDFR_RCM)} AS {tipo_sql(tipo_campo_contrato_hive(contrato['origem_hive'], COL_NR_IDFR_RCM))}) AS {coluna_sql(COL_NR_IDFR_RCM)},
    #         CAST({coluna_sql(COL_NR_VRS_VLDD_RCM)} AS {tipo_sql(tipo_campo_contrato_hive(contrato['origem_hive'], COL_NR_VRS_VLDD_RCM))}) AS {coluna_sql(COL_NR_VRS_VLDD_RCM)},
    #         CAST({coluna_sql(COL_CD_CLI)} AS {tipo_sql(tipo_campo_contrato_hive(contrato['origem_hive'], COL_CD_CLI))}) AS {coluna_sql(COL_CD_CLI)},
    #         CAST({coluna_sql(COL_PC_NVL_CNFA_AVS)} AS {tipo_sql(tipos_oracle[ORACLE_COL_PC_NVL_CNFA_AVS])}) AS {coluna_sql(ORACLE_COL_PC_NVL_CNFA_AVS)},
    #         TRIM(CAST({coluna_sql(COL_CD_IDFR_AVS)} AS {tipo_sql(tipos_oracle[ORACLE_COL_CD_IDFR_AVS])})) AS {coluna_sql(ORACLE_COL_CD_IDFR_AVS)},
    #         CAST({coluna_sql(COL_DT_AVS_FNC_CLI)} AS {tipo_sql(tipos_oracle[ORACLE_COL_DT_AVS_FNC_CLI])}) AS {coluna_sql(ORACLE_COL_DT_AVS_FNC_CLI)},
    #         CAST({coluna_sql(COL_TX_TIT_PDRO_AVS)} AS {tipo_sql(tipos_oracle[ORACLE_COL_TX_TIT_AVS_FNC_CLI])}) AS {coluna_sql(ORACLE_COL_TX_TIT_AVS_FNC_CLI)},
    #         CAST({coluna_sql(COL_TX_STIT_PDRO_AVS)} AS {tipo_sql(tipos_oracle[ORACLE_COL_TX_AVS_FNC_CLI])}) AS {coluna_sql(ORACLE_COL_TX_AVS_FNC_CLI)},
    #         CAST({coluna_sql(COL_TX_PRM_HDR_API_AVS)} AS {tipo_sql(tipos_oracle[ORACLE_COL_TX_PRM_HDR_API_AVS])}) AS {coluna_sql(ORACLE_COL_TX_PRM_HDR_API_AVS)},
    #         CAST({coluna_sql(COL_CD_CLI)} AS {tipo_sql(tipos_oracle[ORACLE_COL_CD_CLI_AVS])}) AS {coluna_sql(ORACLE_COL_CD_CLI_AVS)}
    #     FROM {view_base}
    # """)

    df = spark_sql(f"""
        SELECT
            CAST({coluna_sql(COL_NR_IDFR_PROJ)} AS {tipo_sql(tipo_campo_contrato_hive(contrato['origem_hive'], COL_NR_IDFR_PROJ))}) AS {coluna_sql(COL_NR_IDFR_PROJ)},
            CAST({coluna_sql(COL_NR_IDFR_RCM)} AS {tipo_sql(tipo_campo_contrato_hive(contrato['origem_hive'], COL_NR_IDFR_RCM))}) AS {coluna_sql(COL_NR_IDFR_RCM)},
            CAST({coluna_sql(COL_NR_VRS_VLDD_RCM)} AS {tipo_sql(tipo_campo_contrato_hive(contrato['origem_hive'], COL_NR_VRS_VLDD_RCM))}) AS {coluna_sql(COL_NR_VRS_VLDD_RCM)},
            CAST({coluna_sql(COL_CD_CLI)} AS {tipo_sql(tipo_campo_contrato_hive(contrato['origem_hive'], COL_CD_CLI))}) AS {coluna_sql(COL_CD_CLI)},

            CAST({coluna_sql(COL_PC_NVL_CNFA_AVS)} AS {tipo_sql(tipos_oracle[ORACLE_COL_PC_NVL_CNFA_AVS])}) AS {coluna_sql(ORACLE_COL_PC_NVL_CNFA_AVS)},

            TRIM(CAST({coluna_sql(COL_CD_IDFR_AVS)} AS STRING)) AS {coluna_sql(ORACLE_COL_CD_IDFR_AVS)},

            CAST({coluna_sql(COL_DT_AVS_FNC_CLI)} AS {tipo_sql(tipos_oracle[ORACLE_COL_DT_AVS_FNC_CLI])}) AS {coluna_sql(ORACLE_COL_DT_AVS_FNC_CLI)},

            SUBSTRING(
                TRIM(CAST({coluna_sql(COL_TX_TIT_PDRO_AVS)} AS STRING)),
                1,
                {limites[ORACLE_COL_TX_TIT_AVS_FNC_CLI]["max_caracteres"]}
            ) AS {coluna_sql(ORACLE_COL_TX_TIT_AVS_FNC_CLI)},

            SUBSTRING(
                TRIM(CAST({coluna_sql(COL_TX_STIT_PDRO_AVS)} AS STRING)),
                1,
                {limites[ORACLE_COL_TX_AVS_FNC_CLI]["max_caracteres"]}
            ) AS {coluna_sql(ORACLE_COL_TX_AVS_FNC_CLI)},

            SUBSTRING(
                TRIM(CAST({coluna_sql(COL_TX_PRM_HDR_API_AVS)} AS STRING)),
                1,
                {limites[ORACLE_COL_TX_PRM_HDR_API_AVS]["max_caracteres"]}
            ) AS {coluna_sql(ORACLE_COL_TX_PRM_HDR_API_AVS)},

            CAST({coluna_sql(COL_CD_CLI)} AS {tipo_sql(tipos_oracle[ORACLE_COL_CD_CLI_AVS])}) AS {coluna_sql(ORACLE_COL_CD_CLI_AVS)}

        FROM {view_base}
    """)
        
    validar_nao_nulo(
        df,
        [
            ORACLE_COL_PC_NVL_CNFA_AVS,
            ORACLE_COL_CD_IDFR_AVS,
            ORACLE_COL_DT_AVS_FNC_CLI,
            ORACLE_COL_TX_TIT_AVS_FNC_CLI,
            ORACLE_COL_TX_AVS_FNC_CLI,
            ORACLE_COL_CD_CLI_AVS,
        ],
        "PUBLICACAO_ORACLE_PREPARADO",
        logger_etapa=logger_etapa,
    )
    validar_sem_duplicidade(df, contrato["chaves"], "PUBLICACAO_ORACLE_PREPARADO", logger_etapa=logger_etapa)

    view_preparado = registrar_view_sql(df, "publicacao_oracle_preparado")
    validacoes_limite = [
        {
            "campo": ORACLE_COL_CD_IDFR_AVS,
            "evento": "VALOR_VAZIO",
            "motivo": "CD_IDFR_AVS vazio",
            "condicao": f"{coluna_sql(ORACLE_COL_CD_IDFR_AVS)} = ''",
            "valor_expr": f"CAST({coluna_sql(ORACLE_COL_CD_IDFR_AVS)} AS STRING)",
            "tamanho_expr": f"LENGTH({coluna_sql(ORACLE_COL_CD_IDFR_AVS)})",
            "limite": "NAO_VAZIO",
            "detalhes": ["campo", "valor_encontrado", "tamanho_encontrado", "limite_esperado"],
        },
        {
            "campo": ORACLE_COL_CD_IDFR_AVS,
            "evento": "LIMITE_EXCEDIDO",
            "motivo": "CD_IDFR_AVS acima do limite fisico",
            "condicao": f"LENGTH({coluna_sql(ORACLE_COL_CD_IDFR_AVS)}) > {literal_sql(limites[ORACLE_COL_CD_IDFR_AVS]['max_caracteres'])}",
            "valor_expr": "CAST(NULL AS STRING)",
            "tamanho_expr": f"LENGTH({coluna_sql(ORACLE_COL_CD_IDFR_AVS)})",
            "limite": str(limites[ORACLE_COL_CD_IDFR_AVS]["max_caracteres"]),
            "detalhes": ["campo", "tamanho_encontrado", "limite_esperado"],
        },
        {
            "campo": ORACLE_COL_TX_TIT_AVS_FNC_CLI,
            "evento": "LIMITE_EXCEDIDO",
            "motivo": "TX_TIT_AVS_FNC_CLI acima do limite fisico",
            "condicao": f"LENGTH({coluna_sql(ORACLE_COL_TX_TIT_AVS_FNC_CLI)}) > {literal_sql(limites[ORACLE_COL_TX_TIT_AVS_FNC_CLI]['max_caracteres'])}",
            "valor_expr": "CAST(NULL AS STRING)",
            "tamanho_expr": f"LENGTH({coluna_sql(ORACLE_COL_TX_TIT_AVS_FNC_CLI)})",
            "limite": str(limites[ORACLE_COL_TX_TIT_AVS_FNC_CLI]["max_caracteres"]),
            "detalhes": ["campo", "tamanho_encontrado", "limite_esperado"],
        },
        {
            "campo": ORACLE_COL_TX_AVS_FNC_CLI,
            "evento": "LIMITE_EXCEDIDO",
            "motivo": "TX_AVS_FNC_CLI acima do limite fisico",
            "condicao": f"LENGTH({coluna_sql(ORACLE_COL_TX_AVS_FNC_CLI)}) > {literal_sql(limites[ORACLE_COL_TX_AVS_FNC_CLI]['max_caracteres'])}",
            "valor_expr": "CAST(NULL AS STRING)",
            "tamanho_expr": f"LENGTH({coluna_sql(ORACLE_COL_TX_AVS_FNC_CLI)})",
            "limite": str(limites[ORACLE_COL_TX_AVS_FNC_CLI]["max_caracteres"]),
            "detalhes": ["campo", "tamanho_encontrado", "limite_esperado"],
        },
        {
            "campo": ORACLE_COL_TX_PRM_HDR_API_AVS,
            "evento": "LIMITE_EXCEDIDO",
            "motivo": "TX_PRM_HDR_API_AVS acima do limite fisico",
            "condicao": f"LENGTH({coluna_sql(ORACLE_COL_TX_PRM_HDR_API_AVS)}) > {literal_sql(limites[ORACLE_COL_TX_PRM_HDR_API_AVS]['max_caracteres'])}",
            "valor_expr": "CAST(NULL AS STRING)",
            "tamanho_expr": f"LENGTH({coluna_sql(ORACLE_COL_TX_PRM_HDR_API_AVS)})",
            "limite": str(limites[ORACLE_COL_TX_PRM_HDR_API_AVS]["max_caracteres"]),
            "detalhes": ["campo", "tamanho_encontrado", "limite_esperado"],
        },
        {
            "campo": ORACLE_COL_PC_NVL_CNFA_AVS,
            "evento": "FAIXA_INVALIDA",
            "motivo": "PC_NVL_CNFA_AVS fora da faixa fisica",
            "condicao": f"{coluna_sql(ORACLE_COL_PC_NVL_CNFA_AVS)} < {literal_sql(limites[ORACLE_COL_PC_NVL_CNFA_AVS]['minimo'])} OR {coluna_sql(ORACLE_COL_PC_NVL_CNFA_AVS)} > {literal_sql(limites[ORACLE_COL_PC_NVL_CNFA_AVS]['maximo'])}",
            "valor_expr": f"CAST({coluna_sql(ORACLE_COL_PC_NVL_CNFA_AVS)} AS STRING)",
            "tamanho_expr": "CAST(NULL AS INT)",
            "limite": f"{limites[ORACLE_COL_PC_NVL_CNFA_AVS]['minimo']}..{limites[ORACLE_COL_PC_NVL_CNFA_AVS]['maximo']}",
            "detalhes": ["campo", "valor_encontrado", "limite_esperado"],
        },
        {
            "campo": ORACLE_COL_CD_CLI_AVS,
            "evento": "LIMITE_EXCEDIDO",
            "motivo": "CD_CLI_AVS acima do limite fisico",
            "condicao": f"LENGTH(CAST({coluna_sql(ORACLE_COL_CD_CLI_AVS)} AS STRING)) > {literal_sql(limites[ORACLE_COL_CD_CLI_AVS]['max_digitos'])}",
            "valor_expr": f"CAST({coluna_sql(ORACLE_COL_CD_CLI_AVS)} AS STRING)",
            "tamanho_expr": f"LENGTH(CAST({coluna_sql(ORACLE_COL_CD_CLI_AVS)} AS STRING))",
            "limite": str(limites[ORACLE_COL_CD_CLI_AVS]["max_digitos"]),
            "detalhes": [ORACLE_COL_CD_CLI_AVS, "campo", "valor_encontrado", "tamanho_encontrado", "limite_esperado"],
        },
    ]
    possui_invalido = False

    for validacao in validacoes_limite:
        if existe_sql(f"SELECT 1 AS ERRO FROM {view_preparado} WHERE {validacao['condicao']}"):
            possui_invalido = True
            df_invalidos = spark_sql(f"""
                SELECT
                    *,
                    {literal_sql(validacao['campo'])} AS campo,
                    {validacao['valor_expr']} AS valor_encontrado,
                    {validacao['tamanho_expr']} AS tamanho_encontrado,
                    {literal_sql(validacao['limite'])} AS limite_esperado
                FROM {view_preparado}
                WHERE {validacao['condicao']}
            """)
            registrar_erros_recomendacao(
                df_invalidos,
                logger_etapa,
                "PUBLICACAO_ORACLE_PREPARADO",
                validacao["evento"],
                validacao["motivo"],
                validacao["detalhes"],
            )

    if possui_invalido:
        raise ErroContratoDados(
            codigo="ORACLE_LIMITE_FISICO_EXCEDIDO",
            mensagem="DataFrame preparado possui valor fora dos limites fisicos do Oracle.",
            etapa="PUBLICACAO_ORACLE",
            objeto=tabela_oracle,
            acao="Corrigir os valores indicados nos logs detalhados antes da publicacao Oracle.",
        )

    return projetar_contrato_oracle(df, tabela_oracle)


def publicar_oracle(
    df_rcm_fnc_cli_final,
    database: str,
    oracle_schema: str,
    cliente_oracle,
    executar: bool,
    hive_publicado: bool,
    detalhar_recomendacao: bool = False,
) -> dict:
    logger_etapa = logger
    tabela_oracle = TABELAS_ORACLE["oracle_cliente_recomendacao_ativa"]
    contrato = CONTRATO_ORACLE[tabela_oracle]
    tipos_oracle = {
        nome_coluna: tipo_coluna
        for nome_coluna, tipo_coluna, _ in contrato["campos"]
    }
    owner = validar_identificador_sql(oracle_schema or cliente_oracle.schema, "oracle_schema").upper()

    query_oracle = f"""
        SELECT
            {ORACLE_COL_PC_NVL_CNFA_AVS},
            {ORACLE_COL_CD_IDFR_AVS},
            {ORACLE_COL_DT_AVS_FNC_CLI},
            {ORACLE_COL_TX_TIT_AVS_FNC_CLI},
            {ORACLE_COL_TX_AVS_FNC_CLI},
            {ORACLE_COL_TX_PRM_HDR_API_AVS},
            {ORACLE_COL_CD_CLI_AVS}
        FROM {owner}.{tabela_oracle}
    """

    if executar and not hive_publicado:
        if detalhar_recomendacao:
            logger_etapa.info("\n".join([
                "[DETALHE_AGRUPADO]",
                "",
                "[AÇÃO] BLOQUEIA_SEM_HIVE_PUBLICADO [ORACLE] BLOQUEADO [QTD] 1",
                "",
                "    ORACLE       → publicação bloqueada porque Hive não foi publicado",
            ]))

        raise ErroOperacional(
            codigo="ORACLE_PUBLICACAO_BLOQUEADA_HIVE",
            mensagem="Publicacao Oracle bloqueada porque a publicacao Hive nao foi concluida.",
            etapa="PUBLICACAO_ORACLE",
            objeto=nome_tabela_hive(database, contrato["origem_hive"]),
            acao="Concluir e validar a publicacao Hive antes de publicar no Oracle.",
        )

    if executar:
        df_base = ler_tabela_hive(database, contrato["origem_hive"])
    else:
        df_base = df_rcm_fnc_cli_final

    df_preparado = _preparar_df_oracle(df_base, logger_etapa)
    colunas = [nome for nome, _, _ in contrato["campos"]]
    qtd_hive_preparado = contar_dataframe_sql(
        df_preparado,
        "PUBLICACAO_ORACLE_HIVE_PREPARADO",
    )

    if executar and qtd_hive_preparado == 0:
        raise ErroContratoDados(
            codigo="ORACLE_CARGA_VAZIA",
            mensagem="Publicacao Oracle bloqueada porque o DataFrame preparado esta vazio.",
            etapa="PUBLICACAO_ORACLE",
            objeto=f"{owner}.{tabela_oracle}",
            acao="Verificar a visao Hive publicada e a preparacao dos dados antes da recarga Oracle.",
        )

    df_oracle_atual = cliente_oracle.run_select(query_oracle)
    df_oracle_atual = projetar_contrato_oracle(df_oracle_atual, tabela_oracle)

    tem_diferenca = comparar_dataframes_materialmente(
        df_preparado,
        df_oracle_atual,
        colunas,
        contrato,
    )

    stats = {
        "dry_run": not executar,
        "qtd_hive_preparado": qtd_hive_preparado,
        "qtd_oracle_antes": contar_dataframe_sql(df_oracle_atual, "PUBLICACAO_ORACLE_ANTES"),
        "qtd_oracle_depois": None,
        "tem_diferenca_material": tem_diferenca,
        "carga_executada": False,
        "motivo": None,
    }

    detalhes_publicacao = []
    if detalhar_recomendacao:
        view_detalhe = registrar_view_sql(df_preparado, "publicacao_oracle_detalhe")
        detalhes_publicacao = [
            row.asDict(recursive=True)
            for row in coletar_sql(f"""
                SELECT
                    {coluna_sql(ORACLE_COL_CD_IDFR_AVS)},
                    COUNT(1) AS QTD_CLIENTES
                FROM {view_detalhe}
                GROUP BY {coluna_sql(ORACLE_COL_CD_IDFR_AVS)}
                ORDER BY {coluna_sql(ORACLE_COL_CD_IDFR_AVS)}
            """)
        ]

    if not tem_diferenca:
        stats["motivo"] = "SEM_DIFERENCA_MATERIAL"
        logger_etapa.info("[PUBLICACAO_ORACLE][SEM_DIFERENCA_MATERIAL] Oracle preservado. Nenhum DELETE+append executado.")
        if detalhar_recomendacao and detalhes_publicacao:
            linhas_detalhe = [
                "[DETALHE_AGRUPADO]",
                "",
                f"[AÇÃO] SEM_DIFERENCA_MATERIAL [ORACLE] PRESERVADO [QTD] {len(detalhes_publicacao)}",
                "",
            ]

            for detalhe in detalhes_publicacao:
                linhas_detalhe.append(f"    - AVISO={detalhe.get(ORACLE_COL_CD_IDFR_AVS)}")
                linhas_detalhe.append(f"        CLIENTES     → QTD={detalhe.get('QTD_CLIENTES')}")
                linhas_detalhe.append("        COMPARACAO   → DIFERENCA_MATERIAL=False")
                linhas_detalhe.append("        ORACLE       → preservado; nenhum DELETE+append executado")
                linhas_detalhe.append("")

            logger_etapa.info("\n".join(linhas_detalhe))

        return {
            "stats": stats,
        }

    if not executar:
        stats["motivo"] = "DRY_RUN_COM_DIFERENCA_MATERIAL"
        logger_etapa.info(
            "[PUBLICACAO_ORACLE][DRY_RUN] Diferenca material detectada. DELETE + append seria executado se executar=True.",
        )
        if detalhar_recomendacao and detalhes_publicacao:
            linhas_detalhe = [
                "[DETALHE_AGRUPADO]",
                "",
                f"[AÇÃO] DRY_RUN_COM_DIFERENCA_MATERIAL [ORACLE] SIMULADO [QTD] {len(detalhes_publicacao)}",
                "",
            ]

            for detalhe in detalhes_publicacao:
                linhas_detalhe.append(f"    - AVISO={detalhe.get(ORACLE_COL_CD_IDFR_AVS)}")
                linhas_detalhe.append(f"        CLIENTES     → QTD={detalhe.get('QTD_CLIENTES')}")
                linhas_detalhe.append("        COMPARACAO   → DIFERENCA_MATERIAL=True")
                linhas_detalhe.append("        ORACLE       → DELETE+append seria executado se executar=True")
                linhas_detalhe.append("")

            logger_etapa.info("\n".join(linhas_detalhe))

        return {
            "stats": stats,
        }

    logger_etapa.info(
        "[PUBLICACAO_ORACLE][RISCO_ASSUMIDO] Diferenca material detectada. "
        f"Iniciando DELETE+append na {tabela_oracle}. Operacao nao atomica.",
    )

    cliente_oracle.reload_dataframe(
        df=df_preparado,
        table_name=tabela_oracle,
        owner=owner,
        batchsize=5000,
        num_partitions=1,
        use_truncate=False,
    )

    df_oracle_depois = cliente_oracle.run_select(query_oracle)
    df_oracle_depois = projetar_contrato_oracle(df_oracle_depois, tabela_oracle)

    if comparar_dataframes_materialmente(df_preparado, df_oracle_depois, colunas, contrato):
        view_base_contexto = registrar_view_sql(df_base, "publicacao_oracle_base_contexto")
        df_contexto = spark_sql(f"""
            SELECT DISTINCT
                {coluna_sql(COL_NR_IDFR_PROJ)},
                {coluna_sql(COL_NR_IDFR_RCM)},
                {coluna_sql(COL_NR_VRS_VLDD_RCM)},
                TRIM(CAST({coluna_sql(COL_CD_IDFR_AVS)} AS {tipo_sql(tipos_oracle[ORACLE_COL_CD_IDFR_AVS])})) AS {coluna_sql(ORACLE_COL_CD_IDFR_AVS)},
                CAST({coluna_sql(COL_CD_CLI)} AS {tipo_sql(tipo_campo_contrato_hive(contrato['origem_hive'], COL_CD_CLI))}) AS {coluna_sql(COL_CD_CLI)},
                CAST({coluna_sql(COL_CD_CLI)} AS {tipo_sql(tipos_oracle[ORACLE_COL_CD_CLI_AVS])}) AS {coluna_sql(ORACLE_COL_CD_CLI_AVS)}
            FROM {view_base_contexto}
        """)
        view_contexto = registrar_view_sql(df_contexto, "publicacao_oracle_contexto")
        view_hive = registrar_view_sql(df_preparado, "publicacao_oracle_hive_pos")
        view_oracle = registrar_view_sql(df_oracle_depois, "publicacao_oracle_depois_pos")
        cond_chaves = (
            f"hive.{coluna_sql(ORACLE_COL_CD_IDFR_AVS)} <=> oracle.{coluna_sql(ORACLE_COL_CD_IDFR_AVS)} "
            f"AND hive.{coluna_sql(ORACLE_COL_CD_CLI_AVS)} <=> oracle.{coluna_sql(ORACLE_COL_CD_CLI_AVS)}"
        )
        cond_contexto = (
            f"hive.{coluna_sql(ORACLE_COL_CD_IDFR_AVS)} <=> ctx.{coluna_sql(ORACLE_COL_CD_IDFR_AVS)} "
            f"AND hive.{coluna_sql(ORACLE_COL_CD_CLI_AVS)} <=> ctx.{coluna_sql(ORACLE_COL_CD_CLI_AVS)}"
        )
        selects_divergencia = [
            f"""
            SELECT
                ctx.{coluna_sql(COL_NR_IDFR_PROJ)} AS {coluna_sql(COL_NR_IDFR_PROJ)},
                ctx.{coluna_sql(COL_NR_IDFR_RCM)} AS {coluna_sql(COL_NR_IDFR_RCM)},
                ctx.{coluna_sql(COL_NR_VRS_VLDD_RCM)} AS {coluna_sql(COL_NR_VRS_VLDD_RCM)},
                hive.{coluna_sql(ORACLE_COL_CD_IDFR_AVS)} AS {coluna_sql(ORACLE_COL_CD_IDFR_AVS)},
                ctx.{coluna_sql(COL_CD_CLI)} AS {coluna_sql(COL_CD_CLI)},
                hive.{coluna_sql(ORACLE_COL_CD_CLI_AVS)} AS {coluna_sql(ORACLE_COL_CD_CLI_AVS)},
                {literal_sql('__LINHA__')} AS campo,
                {literal_sql('PRESENTE_HIVE')} AS valor_hive,
                {literal_sql('AUSENTE_ORACLE')} AS valor_oracle,
                {literal_sql('AUSENTE_ORACLE')} AS tipo_divergencia
            FROM {view_hive} hive
            LEFT JOIN {view_oracle} oracle
                ON {cond_chaves}
            INNER JOIN {view_contexto} ctx
                ON {cond_contexto}
            WHERE oracle.{coluna_sql(ORACLE_COL_CD_IDFR_AVS)} IS NULL
            """
        ]
        selects_divergencia.extend(
            f"""
            SELECT
                ctx.{coluna_sql(COL_NR_IDFR_PROJ)} AS {coluna_sql(COL_NR_IDFR_PROJ)},
                ctx.{coluna_sql(COL_NR_IDFR_RCM)} AS {coluna_sql(COL_NR_IDFR_RCM)},
                ctx.{coluna_sql(COL_NR_VRS_VLDD_RCM)} AS {coluna_sql(COL_NR_VRS_VLDD_RCM)},
                hive.{coluna_sql(ORACLE_COL_CD_IDFR_AVS)} AS {coluna_sql(ORACLE_COL_CD_IDFR_AVS)},
                ctx.{coluna_sql(COL_CD_CLI)} AS {coluna_sql(COL_CD_CLI)},
                hive.{coluna_sql(ORACLE_COL_CD_CLI_AVS)} AS {coluna_sql(ORACLE_COL_CD_CLI_AVS)},
                {literal_sql(coluna)} AS campo,
                CAST(hive.{coluna_sql(coluna)} AS STRING) AS valor_hive,
                CAST(oracle.{coluna_sql(coluna)} AS STRING) AS valor_oracle,
                {literal_sql('CAMPO_DIVERGENTE')} AS tipo_divergencia
            FROM {view_hive} hive
            INNER JOIN {view_oracle} oracle
                ON {cond_chaves}
            INNER JOIN {view_contexto} ctx
                ON {cond_contexto}
            WHERE NOT (hive.{coluna_sql(coluna)} <=> oracle.{coluna_sql(coluna)})
            """
            for coluna in colunas
        )
        df_divergencias_hive = spark_sql("\nUNION ALL\n".join(selects_divergencia))
        registrar_erros_recomendacao(
            df_divergencias_hive,
            logger_etapa,
            "PUBLICACAO_ORACLE_POS_CARGA",
            "DIVERGENCIA_POS_CARGA",
            "Oracle pos-carga diverge do Hive preparado",
            [ORACLE_COL_CD_CLI_AVS, "campo", "valor_hive", "valor_oracle", "tipo_divergencia"],
        )
        linhas_extra_oracle = coletar_sql(f"""
            SELECT DISTINCT
                oracle.{coluna_sql(ORACLE_COL_CD_IDFR_AVS)} AS {coluna_sql(ORACLE_COL_CD_IDFR_AVS)},
                oracle.{coluna_sql(ORACLE_COL_CD_CLI_AVS)} AS {coluna_sql(ORACLE_COL_CD_CLI_AVS)}
            FROM {view_oracle} oracle
            LEFT JOIN {view_hive} hive
                ON {cond_chaves}
            WHERE hive.{coluna_sql(ORACLE_COL_CD_IDFR_AVS)} IS NULL
        """)
        for linha_extra in linhas_extra_oracle:
            logger_etapa.error(
                "[PUBLICACAO_ORACLE_POS_CARGA][LINHA_EXTRA_ORACLE] "
                f"CD_IDFR_AVS={linha_extra[ORACLE_COL_CD_IDFR_AVS]} "
                f"CD_CLI_AVS={linha_extra[ORACLE_COL_CD_CLI_AVS]}"
            )
        raise ErroContratoDados(
            codigo="ORACLE_POS_CARGA_DIVERGENTE",
            mensagem="Oracle pos-carga diverge do Hive preparado.",
            etapa="PUBLICACAO_ORACLE",
            objeto=f"{owner}.{tabela_oracle}",
            acao="Analisar as divergencias detalhadas e reconciliar a tabela Oracle antes de nova execucao.",
        )

    stats["qtd_oracle_depois"] = contar_dataframe_sql(df_oracle_depois, "PUBLICACAO_ORACLE_DEPOIS")
    stats["carga_executada"] = True
    stats["motivo"] = "CARGA_EXECUTADA_DELETE_APPEND"

    if detalhar_recomendacao and detalhes_publicacao:
        linhas_detalhe = [
            "[DETALHE_AGRUPADO]",
            "",
            f"[AÇÃO] CARGA_EXECUTADA_DELETE_APPEND [ORACLE] EXECUTADO [QTD] {len(detalhes_publicacao)}",
            "",
        ]

        for detalhe in detalhes_publicacao:
            linhas_detalhe.append(f"    - AVISO={detalhe.get(ORACLE_COL_CD_IDFR_AVS)}")
            linhas_detalhe.append(f"        CLIENTES     → QTD={detalhe.get('QTD_CLIENTES')}")
            linhas_detalhe.append("        COMPARACAO   → DIFERENCA_MATERIAL=True")
            linhas_detalhe.append("        ORACLE       → DELETE+append executado")
            linhas_detalhe.append("")

        logger_etapa.info("\n".join(linhas_detalhe))

    logger_etapa.obj(stats, title="[PUBLICACAO_ORACLE][FIM] Stats")

    return {
        "stats": stats,
    }